In [ ]:
# ========== 全局导入：OpenAI / Chroma / 环境变量 ==========

# 类型标注：可空与列表
from typing import Optional, List
# OpenAI 客户端：后续 Scanner / Frontier / Planning 共用
from openai import OpenAI
# JSON：工具参数与 memory 序列化
import json
# 从 dotenv 导入 load_dotenv：把 .env 密钥读进环境变量
from dotenv import load_dotenv
# Chroma：产品向量库（RAG 可比商品）
import chromadb
# 标准库日志与操作系统环境
import logging
import os

# override=True：.env 覆盖进程里已有同名变量
load_dotenv(override=True)
# 默认用 OPENAI_API_KEY 建客户端
openai = OpenAI()
# Planning / Messaging 等处引用的主模型 id（保持原样）
MODEL = "gpt-5.4"


In [ ]:
# ========== Modal 远程 Pricer：4bit 量化 Llama + LoRA 微调权重 ==========

# Modal：把估价模型部署到云端 GPU
import modal
from modal import Volume, Image

# App 名需与 SpecialistAgent.from_name 一致
app = modal.App("pricer-service")
# 精简 Debian 镜像 + 推理依赖
image = Image.debian_slim().pip_install(
    "huggingface", "torch", "transformers", "bitsandbytes", "accelerate", "peft"
)

# 从 Modal 拉取名为 huggingface-secret 的密钥（HF token）
# 若你的 Modal 配置不同，可能需改成 hf-secret
secrets = [modal.Secret.from_name("huggingface-secret")]

# GPU 型号、基座模型、微调仓库坐标（字符串保持原样）
GPU = "T4"
BASE_MODEL = "meta-llama/Llama-3.2-3B"
PROJECT_NAME = "price"
HF_USER = "denis-mutuma"  # your HF name here! Or use mine if you just want to reproduce my results.
RUN_NAME = "2026-03-08_13.42.08-lite"
PROJECT_RUN_NAME = f"{PROJECT_NAME}-{RUN_NAME}"
FINETUNED_MODEL = f"{HF_USER}/{PROJECT_RUN_NAME}"
REVISION = "main"  # HuggingFace model revision (e.g. "main" or a commit hash)
CACHE_DIR = "/cache"

# min_containers=0：空闲后缩容；改成 1 可常驻热容器
MIN_CONTAINERS = 0

# 生成 prompt 的固定前后缀（影响解析，勿改）
PREFIX = "Price is $"
QUESTION = "What does this cost to the nearest dollar?"

# Hugging Face hub 缓存卷：跨调用复用下载
hf_cache_volume = Volume.from_name("hf-hub-cache", create_if_missing=True)

@app.cls(
    image=image.env({"HF_HUB_CACHE": CACHE_DIR}),
    secrets=secrets,
    gpu=GPU,
    timeout=1800,
    min_containers=MIN_CONTAINERS,
    volumes={CACHE_DIR: hf_cache_volume},
)
class Pricer:
    @modal.enter()
    def setup(self):
        # 容器冷启动时加载一次模型
        import torch
        from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
        from peft import PeftModel

        # 4bit NF4 量化配置：省显存
        quant_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4",
        )

        # 基座 tokenizer + 因果 LM；再挂上 LoRA/PEFT 权重
        self.tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
        self.tokenizer.pad_token = self.tokenizer.eos_token
        self.tokenizer.padding_side = "right"
        self.base_model = AutoModelForCausalLM.from_pretrained(
            BASE_MODEL, quantization_config=quant_config, device_map="auto"
        )
        self.fine_tuned_model = PeftModel.from_pretrained(
            self.base_model, FINETUNED_MODEL, revision=REVISION
        )

    @modal.method()
    def price(self, description: str) -> float:
        # 远程方法：描述 → 生成短续写 → 正则抠价格
        import re
        import torch
        from transformers import set_seed

        set_seed(42)
        prompt = f"{QUESTION}\n\n{description}\n\n{PREFIX}"

        inputs = self.tokenizer.encode(prompt, return_tensors="pt").to("cuda")
        with torch.no_grad():
            outputs = self.fine_tuned_model.generate(inputs, max_new_tokens=5)
        result = self.tokenizer.decode(outputs[0])
        # 按 PREFIX 切开，右侧应是数字价格
        contents = result.split("Price is $")[1]
        contents = contents.replace(",", "")
        match = re.search(r"[-+]?\d*\.\d+|\d+", contents)
        return float(match.group()) if match else 0


In [ ]:
# ========== Planning 用的 system/user 种子消息（英文 prompt 不翻译） ==========

# system：角色 = 找特价并用工具通知
system_message = "You find great deals on bargain products using your tools, and notify the user of the best bargain."
# user：规定工具调用顺序：扫网 → 估价 → 挑最优 → 通知 → 回复 OK
user_message = """
First, use your tool to scan the internet for bargain deals. Then for each deal, use your tool to estimate its true value.
Then pick the single most compelling deal where the price is much lower than the estimated true value, and use your tool to notify the user.
Then just reply OK to indicate success.
"""
# OpenAI messages 列表格式
messages = [{"role": "system", "content": system_message},{"role": "user", "content": user_message}]

# 笔记本里直接展示，便于检查内容
messages


In [ ]:
# ========== 打开根 logger 的 INFO 级别，方便看各 Agent 彩色日志 ==========

root = logging.getLogger()
root.setLevel(logging.INFO)


In [ ]:
# ========== 连接本地 Chroma：products 集合（Frontier / Ensemble RAG） ==========

# 持久化目录名
DB = "products_vectorstore"
# PersistentClient：数据落在磁盘 path 下
client = chromadb.PersistentClient(path=DB)
# 没有就创建名为 products 的 collection
collection = client.get_or_create_collection('products')


In [ ]:
# ========== Agent 基类：ANSI 彩色日志，方便多代理并行时区分来源 ==========

class Agent:
    """Agent 的抽象超类
    用于以可以识别每个代理的方式记录消息"""

    # 前景色（Foreground）ANSI
    RED = '\033[31m'
    GREEN = '\033[32m'
    YELLOW = '\033[33m'
    BLUE = '\033[34m'
    MAGENTA = '\033[35m'
    CYAN = '\033[36m'
    WHITE = '\033[37m'
    
    # 背景颜色：黑底衬托前景
    BG_BLACK = '\033[40m'
    
    # 重置代码：回到终端默认颜色
    RESET = '\033[0m'

    # 子类覆盖：显示名与主题色
    name: str = ""
    color: str = '\033[37m'

    def log(self, message):
        """将此记录为信息消息，用于识别代理"""
        # 黑底 + 代理色 + [name] 前缀
        color_code = self.BG_BLACK + self.color
        message = f"[{self.name}] {message}"
        logging.info(color_code + message + self.RESET)


In [ ]:
# ========== 数据模型 + RSS 抓取：ScrapedDeal / Deal / Opportunity ==========

# Pydantic：结构化输出与校验；Self 用于类方法返回类型
from pydantic import BaseModel, Field
from typing import List, Dict, Self
# BeautifulSoup：清理 HTML 片段
from bs4 import BeautifulSoup
import re
# feedparser：解析 DealNews RSS
import feedparser
from tqdm import tqdm
import requests
import time

# 要订阅的 RSS 源（URL 保持原样）
feeds = [
    "https://www.dealnews.com/c142/Electronics/?rss=1",
    "https://www.dealnews.com/c39/Computers/?rss=1",
    "https://www.dealnews.com/f1912/Smart-Home/?rss=1",
]

# 您还可以添加："https://www.dealnews.com/c238/Automotive/?rss=1"
# "https://www.dealnews.com/c196/Home-Garden/?rss=1"


def extract(html_snippet: str) -> str:
    """使用 Beautiful Soup 清理此 HTML 片段并提取有用的文本"""
    soup = BeautifulSoup(html_snippet, "html.parser")
    # DealNews 摘要常包在 class=snippet summary 的 div 里
    snippet_div = soup.find("div", class_="snippet summary")

    if snippet_div:
        description = snippet_div.get_text(strip=True)
        # 再解析一层，去掉残留实体/标签
        description = BeautifulSoup(description, "html.parser").get_text()
        description = re.sub("<[^<]+?>", "", description)
        result = description.strip()
    else:
        # 找不到结构时退回原始片段
        result = html_snippet
    return result.replace("\n", " ")


class ScrapedDeal:
    """表示从 RSS feed 检索到的 Deal 的类"""

    category: str
    title: str
    summary: str
    url: str
    details: str
    features: str

    def __init__(self, entry: Dict[str, str]):
        """根据提供的字典填充此实例"""
        self.title = entry["title"]
        self.summary = extract(entry["summary"])
        # feedparser 的 links[0].href 是详情页
        self.url = entry["links"][0]["href"]
        # 再请求详情页，拆 Details / Features
        stuff = requests.get(self.url).content
        soup = BeautifulSoup(stuff, "html.parser")
        content = soup.find("div", class_="content-section").get_text()
        content = content.replace("\nmore", "").replace("\n", " ")
        if "Features" in content:
            self.details, self.features = content.split("Features", 1)
        else:
            self.details = content
            self.features = ""
        self.truncate()

    def truncate(self):
        """将字段限制在合理的长度，以避免向模型发送太多信息"""
        self.title = self.title[:100]
        self.details = self.details[:500]
        self.features = self.features[:500]

    def __repr__(self):
        """返回一个字符串来描述这笔交易"""
        return f"<{self.title}>"

    def describe(self):
        """返回一个较长的字符串来描述此交易以用于调用模型"""
        return f"Title: {self.title}\nDetails: {self.details.strip()}\nFeatures: {self.features.strip()}\nURL: {self.url}"

    @classmethod
    def fetch(cls, show_progress: bool = False) -> List[Self]:
        """从选定的 RSS 源检索所有交易"""
        deals = []
        feed_iter = tqdm(feeds) if show_progress else feeds
        for feed_url in feed_iter:
            feed = feedparser.parse(feed_url)
            # 每个源只取前 10 条，控制抓取量
            for entry in feed.entries[:10]:
                deals.append(cls(entry))
                time.sleep(0.05)
        return deals


class Deal(BaseModel):
    """代表交易并带有摘要描述的类"""

    # Field description 会进 Structured Outputs schema，必须保持英文
    product_description: str = Field(
        description="Your clearly expressed summary of the product in 3-4 sentences. Details of the item are much more important than why it's a good deal. Avoid mentioning discounts and coupons; focus on the item itself. There should be a short paragraph of text for each item you choose."
    )
    price: float = Field(
        description="The actual price of this product, as advertised in the deal. Be sure to give the actual price; for example, if a deal is described as $100 off the usual $300 price, you should respond with $200"
    )
    url: str = Field(description="The URL of the deal, as provided in the input")


class DealSelection(BaseModel):
    """代表交易列表的类"""

    deals: List[Deal] = Field(
        description="Your selection of the 5 deals that have the most detailed, high quality description and the most clear price. You should be confident that the price reflects the deal, that it is a good deal, with a clear description"
    )


class Opportunity(BaseModel):
    """代表可能机会的类：我们估计的交易
    它的价格应该比所提供的要高"""

    deal: Deal
    estimate: float
    discount: float



In [ ]:
# ========== ScannerAgent：RSS 抓取 + Structured Outputs 精选 5 笔 ==========

class ScannerAgent(Agent):
    # 扫描阶段用的模型 id（保持原样）
    MODEL = "gpt-5-mini"

    # SYSTEM / USER prompt 英文原样：决定如何挑 deal、如何写 product_description
    SYSTEM_PROMPT = """You identify and summarize the 5 most detailed deals from a list, by selecting deals that have the most detailed, high quality description and the most clear price.
    Respond strictly in JSON with no explanation, using this format. You should provide the price as a number derived from the description. If the price of a deal isn't clear, do not include that deal in your response.
    Most important is that you respond with the 5 deals that have the most detailed product description with price. It's not important to mention the terms of the deal; most important is a thorough description of the product.
    Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 
    """

    USER_PROMPT_PREFIX = """Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price that is greater than 0.
    You should rephrase the description to be a summary of the product itself, not the terms of the deal.
    Remember to respond with a short paragraph of text in the product_description field for each of the 5 items that you select.
    Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 
    
    Deals:
    
    """

    USER_PROMPT_SUFFIX = "\n\nInclude exactly 5 deals, no more."

    name = "Scanner Agent"
    color = Agent.CYAN

    def __init__(self):
        self.log("Scanner Agent is initializing")
        # 新建 OpenAI 客户端（读环境变量里的 Key）
        self.openai = OpenAI()
        self.log("Scanner Agent is ready")

    def fetch_deals(self, memory) -> List[ScrapedDeal]:
        # memory 里已有机会的 URL：本轮跳过，避免重复推荐
        urls = [opp.deal.url for opp in memory]
        scraped = ScrapedDeal.fetch()
        result = [scrape for scrape in scraped if scrape.url not in urls]
        self.log(f"Scanner Agent received {len(result)} deals not already scraped")
        return result

    def make_user_prompt(self, scraped) -> str:
        # 前缀 + 每条 describe() + 后缀
        user_prompt = self.USER_PROMPT_PREFIX
        user_prompt += "\n\n".join([scrape.describe() for scrape in scraped])
        user_prompt += self.USER_PROMPT_SUFFIX
        return user_prompt

    def scan(self, memory: List[str] = []) -> Optional[DealSelection]:
        scraped = self.fetch_deals(memory)
        if scraped:
            user_prompt = self.make_user_prompt(scraped)
            self.log("Scanner Agent is calling OpenAI using Structured Outputs")
            # parse + response_format=DealSelection：强制结构化
            result = self.openai.chat.completions.parse(
                model=self.MODEL,
                messages=[
                    {"role": "system", "content": self.SYSTEM_PROMPT},
                    {"role": "user", "content": user_prompt},
                ],
                response_format=DealSelection,
                reasoning_effort="minimal",
            )
            result = result.choices[0].message.parsed
            # 再过滤掉 price<=0 的脏数据
            result.deals = [deal for deal in result.deals if deal.price > 0]
            self.log(
                f"Scanner Agent received {len(result.deals)} selected deals with price>0 from OpenAI"
            )
            return result
        return None

    def test_scan(self, memory: List[str] = []) -> Optional[DealSelection]:
        # 离线联调用的固定 4 笔测试交易（字符串保持原样）
        results = {
            "deals": [
                {
                    "product_description": "The Hisense R6 Series 55R6030N is a 55-inch 4K UHD Roku Smart TV that offers stunning picture quality with 3840x2160 resolution. It features Dolby Vision HDR and HDR10 compatibility, ensuring a vibrant and dynamic viewing experience. The TV runs on Roku's operating system, allowing easy access to streaming services and voice control compatibility with Google Assistant and Alexa. With three HDMI ports available, connecting multiple devices is simple and efficient.",
                    "price": 178,
                    "url": "https://www.dealnews.com/products/Hisense/Hisense-R6-Series-55-R6030-N-55-4-K-UHD-Roku-Smart-TV/484824.html?iref=rss-c142",
                },
                {
                    "product_description": "The Poly Studio P21 is a 21.5-inch LED personal meeting display designed specifically for remote work and video conferencing. With a native resolution of 1080p, it provides crystal-clear video quality, featuring a privacy shutter and stereo speakers. This display includes a 1080p webcam with manual pan, tilt, and zoom control, along with an ambient light sensor to adjust the vanity lighting as needed. It also supports 5W wireless charging for mobile devices, making it an all-in-one solution for home offices.",
                    "price": 30,
                    "url": "https://www.dealnews.com/products/Poly-Studio-P21-21-5-1080-p-LED-Personal-Meeting-Display/378335.html?iref=rss-c39",
                },
                {
                    "product_description": "The Lenovo IdeaPad Slim 5 laptop is powered by a 7th generation AMD Ryzen 5 8645HS 6-core CPU, offering efficient performance for multitasking and demanding applications. It features a 16-inch touch display with a resolution of 1920x1080, ensuring bright and vivid visuals. Accompanied by 16GB of RAM and a 512GB SSD, the laptop provides ample speed and storage for all your files. This model is designed to handle everyday tasks with ease while delivering an enjoyable user experience.",
                    "price": 446,
                    "url": "https://www.dealnews.com/products/Lenovo/Lenovo-Idea-Pad-Slim-5-7-th-Gen-Ryzen-5-16-Touch-Laptop/485068.html?iref=rss-c39",
                },
                {
                    "product_description": "The Dell G15 gaming laptop is equipped with a 6th-generation AMD Ryzen 5 7640HS 6-Core CPU, providing powerful performance for gaming and content creation. It features a 15.6-inch 1080p display with a 120Hz refresh rate, allowing for smooth and responsive gameplay. With 16GB of RAM and a substantial 1TB NVMe M.2 SSD, this laptop ensures speedy performance and plenty of storage for games and applications. Additionally, it includes the Nvidia GeForce RTX 3050 GPU for enhanced graphics and gaming experiences.",
                    "price": 650,
                    "url": "https://www.dealnews.com/products/Dell/Dell-G15-Ryzen-5-15-6-Gaming-Laptop-w-Nvidia-RTX-3050/485067.html?iref=rss-c39",
                },
            ]
        }
        return DealSelection(**results)


In [ ]:
# ========== SpecialistAgent：远程调用 Modal 上的微调 Pricer ==========

class SpecialistAgent(Agent):
    """运行我们微调的 LLM 的代理，该代理在 Modal 上远程运行"""

    name = "Specialist Agent"
    color = Agent.RED

    def __init__(self):
        """通过创建模态类的实例来设置此代理"""
        self.log("Specialist Agent is initializing - connecting to modal")
        # 按 App 名 / Class 名取得已部署的 Pricer
        Pricer = modal.Cls.from_name("pricer-service", "Pricer")
        self.pricer = Pricer()

    def price(self, description: str) -> float:
        """进行远程调用以返回该商品的预估价格"""
        self.log("Specialist Agent is calling remote fine-tuned model")
        # .remote：在 Modal 容器里执行 price 方法
        result = self.pricer.price.remote(description)
        self.log(f"Specialist Agent completed - predicting ${result:.2f}")
        return result


In [ ]:
# ========== FrontierAgent：Chroma RAG 相似品 + 前沿 LLM 估价 ==========

import re
from typing import List, Dict
from openai import OpenAI
from sentence_transformers import SentenceTransformer

# 在慢速/不可靠的网络上从 Hugging Face 加载时避免 ReadTimeout
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "120")
os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "60")
os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "60")


class FrontierAgent(Agent):
    name = "Frontier Agent"
    color = Agent.BLUE

    # 类属性默认；__init__ 里会改成 gpt-5.1
    MODEL = "gpt-4o-mini"

    def __init__(self, collection):
        """通过连接到 OpenAI 或 DeepSeek、Chroma 数据存储来设置此实例，
        并建立矢量编码模型"""
        self.log("Initializing Frontier Agent")
        self.client = OpenAI()
        self.MODEL = "gpt-5.1"
        self.log("Frontier Agent is setting up with OpenAI")
        # 注入的 Chroma collection（products）
        self.collection = collection
        # MiniLM：描述 → 向量，供 query
        self.model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
        self.log("Frontier Agent is ready")

    def make_context(self, similars: List[str], prices: List[float]) -> str:
        """创建可以插入提示中的上下文
        :paramsimilars: 与被估计的产品相似的产品
        :paramprices: 同类产品的价格
        :return: 在提供上下文的提示中插入的文本"""
        # 英文模板保持原样（会进 user prompt）
        message = "To provide some context, here are some other items that might be similar to the item you need to estimate.\n\n"
        for similar, price in zip(similars, prices):
            message += f"Potentially related product:\n{similar}\nPrice is ${price:.2f}\n\n"
        return message

    def messages_for(
        self, description: str, similars: List[str], prices: List[float]
    ) -> List[Dict[str, str]]:
        """
        Create the message list to be included in a call to OpenAI
        With the system and user prompt
        :param description: a description of the product
        :param similars: similar products to this one
        :param prices: prices of similar products
        :return: the list of messages in the format expected by OpenAI
        """
        # 估价指令 + 描述 + RAG 上下文（英文 prompt 不翻译）
        message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{description}\n\n"
        message += self.make_context(similars, prices)
        return [{"role": "user", "content": message}]

    def find_similars(self, description: str):
        """通过在 Chroma 数据存储中查找，返回与给定项目类似的项目列表"""
        self.log(
            "Frontier Agent is performing a RAG search of the Chroma datastore to find 5 similar products"
        )
        vector = self.model.encode([description])
        # query_embeddings 要 list[list[float]]
        results = self.collection.query(query_embeddings=vector.astype(float).tolist(), n_results=5)
        documents = results["documents"][0][:]
        prices = [m["price"] for m in results["metadatas"][0][:]]
        self.log("Frontier Agent has found similar products")
        return documents, prices

    def get_price(self, s) -> float:
        """从字符串中提取浮点数的实用程序"""
        s = s.replace("$", "").replace(",", "")
        match = re.search(r"[-+]?\d*\.\d+|\d+", s)
        return float(match.group()) if match else 0.0

    def price(self, description: str) -> float:
        """致电 OpenAI 或 DeepSeek 估算所述产品的价格，
        通过查找 5 个类似产品并将它们包含在提示中以提供上下文
        :param description: 产品的描述
        :return: 预估价格"""
        documents, prices = self.find_similars(description)
        self.log(
            f"Frontier Agent is about to call {self.MODEL} with context including 5 similar products"
        )
        response = self.client.chat.completions.create(
            model=self.MODEL,
            messages=self.messages_for(description, documents, prices),
            seed=42,
            reasoning_effort="none",
        )
        reply = response.choices[0].message.content
        result = self.get_price(reply)
        self.log(f"Frontier Agent completed - predicting ${result:.2f}")
        return result



In [ ]:
# ========== 深度残差网络 + 推理封装（HashingVectorizer → 价格） ==========

import numpy as np
from tqdm.notebook import tqdm
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torch.optim.lr_scheduler import CosineAnnealingLR
# HashingVectorizer：文本 → 固定维稀疏二值特征（无需拟合词表）
from sklearn.feature_extraction.text import HashingVectorizer

class ResidualBlock(nn.Module):
    def __init__(self, hidden_size, dropout_prob):
        super(ResidualBlock, self).__init__()
        # 两层 Linear + LayerNorm + Dropout；外面再加 skip
        self.block = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.LayerNorm(hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout_prob),
            nn.Linear(hidden_size, hidden_size),
            nn.LayerNorm(hidden_size),
        )
        self.relu = nn.ReLU()

    def forward(self, x):
        residual = x
        out = self.block(x)
        out += residual  # Skip connection：缓解深层梯度消失
        return self.relu(out)


class DeepNeuralNetwork(nn.Module):
    def __init__(self, input_size, num_layers=10, hidden_size=4096, dropout_prob=0.2):
        super(DeepNeuralNetwork, self).__init__()

        # 第一层：抬升到 hidden_size
        self.input_layer = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.LayerNorm(hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout_prob),
        )

        # 中间堆叠 residual blocks（层数 = num_layers-2）
        self.residual_blocks = nn.ModuleList()
        for i in range(num_layers - 2):
            self.residual_blocks.append(ResidualBlock(hidden_size, dropout_prob))

        # 输出层：回归一个标量（对数空间价格）
        self.output_layer = nn.Linear(hidden_size, 1)

    def forward(self, x):
        x = self.input_layer(x)

        for block in self.residual_blocks:
            x = block(x)

        return self.output_layer(x)


# 训练时对 log(price+1) 做标准化用的均值/方差（反变换要用）
Y_STD = 1.0328539609909058
Y_MEAN = 4.434937953948975


class DeepNeuralNetworkInference:
    def __init__(self):
        self.vectorizer = None
        self.model = None
        self.device = None

        # 固定随机种子，便于复现
        np.random.seed(42)
        torch.manual_seed(42)
        torch.cuda.manual_seed(42)

    def setup(self):
        # n_features=5000 与 DeepNeuralNetwork(5000) 对齐
        self.vectorizer = HashingVectorizer(n_features=5000, stop_words="english", binary=True)
        self.model = DeepNeuralNetwork(5000)
        # 设备优先级：CUDA → MPS → CPU
        if torch.cuda.is_available():
            self.device = torch.device("cuda")
        elif torch.backends.mps.is_available():
            self.device = torch.device("mps")
        else:
            self.device = torch.device("cpu")

        logging.info(f"Neural Network is using {self.device}")

        self.model.to(self.device)

    def load(self, path):
        # map_location：在无 GPU 机器上也能加载权重
        self.model.load_state_dict(torch.load(path, map_location=self.device))
        self.model.to(self.device)

    def inference(self, text):
        self.model.eval()
        with torch.no_grad():
            vector = self.vectorizer.transform([text])
            vector = torch.FloatTensor(vector.toarray()).to(self.device)
            pred = self.model(vector)[0]
            # 反标准化再 expm1：回到美元价格
            result = torch.exp(pred * Y_STD + Y_MEAN) - 1
            result = result.item()
        return max(0, result)



In [ ]:
# ========== NeuralNetworkAgent：加载 deep_neural_network.pth 做本地估价 ==========

class NeuralNetworkAgent(Agent):
    name = "Neural Network Agent"
    color = Agent.MAGENTA

    def __init__(self):
        """通过加载保存的模型权重来初始化该对象
        和 SentenceTransformer 向量编码模型"""
        self.log("Neural Network Agent is initializing")
        self.neural_network = DeepNeuralNetworkInference()
        self.neural_network.setup()
        # 权重文件需在工作目录（或改路径）
        self.neural_network.load("deep_neural_network.pth")
        self.log("Neural Network Agent is ready and weights are loaded")

    def price(self, description: str) -> float:
        """使用深度神经网络来估计所描述商品的价格
        :param 描述: 待估算的产品
        :return: 浮动价格"""
        self.log("Neural Network Agent is starting a prediction")
        result = self.neural_network.inference(description)
        self.log(f"Neural Network Agent completed - predicting ${result:.2f}")
        return result



In [ ]:
# ========== Preprocessor：用 LiteLLM 把杂乱描述改写成统一短格式 ==========

from litellm import completion

# 默认值：OpenAI（无需 Ollama 即可工作）。对于本地 Ollama，请使用 PRICER_PREPROCESSOR_MODEL=ollama/llama3.2。
DEFAULT_MODEL_NAME = os.getenv("PRICER_PREPROCESSOR_MODEL", "openai/gpt-4o-mini")
# gpt-oss 系列才传 reasoning_effort；其它模型为 None
DEFAULT_REASONING_EFFORT = "low" if "gpt-oss" in DEFAULT_MODEL_NAME else None

# 改写格式约束（英文 prompt 不翻译）
SYSTEM_PROMPT = """Create a concise description of a product. Respond only in this format. Do not include part numbers.
Title: Rewritten short precise title
Category: eg Electronics
Brand: Brand name
Description: 1 sentence description
Details: 1 sentence on features"""


class Preprocessor:
    def __init__(
        self,
        model_name=DEFAULT_MODEL_NAME,
        reasoning_effort=DEFAULT_REASONING_EFFORT,
        base_url=None,
    ):
        # 累计 token / 费用，便于观察预处理开销
        self.total_input_tokens = 0
        self.total_output_tokens = 0
        self.total_cost = 0
        self.model_name = model_name
        self.reasoning_effort = reasoning_effort
        self.base_url = base_url
        # Ollama 默认本地地址
        if "ollama" in model_name and not base_url:
            self.base_url = "http://localhost:11434"

    def messages_for(self, text: str) -> list[dict]:
        return [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": text}]

    def preprocess(self, text: str) -> str:
        messages = self.messages_for(text)
        response = completion(
            messages=messages,
            model=self.model_name,
            reasoning_effort=self.reasoning_effort,
            api_base=self.base_url,
        )
        # 累加 usage；cost 从 litellm 隐藏字段取
        self.total_input_tokens += response.usage.prompt_tokens
        self.total_output_tokens += response.usage.completion_tokens
        self.total_cost += response._hidden_params["response_cost"]
        return response.choices[0].message.content



In [ ]:
# ========== EnsembleAgent：预处理后加权融合三路估价 ==========

class EnsembleAgent(Agent):
    name = "Ensemble Agent"
    color = Agent.YELLOW

    def __init__(self, collection):
        """通过创建每个模型来创建 Ensemble 的实例
        并加载 Ensemble 的权重"""
        self.log("Initializing Ensemble Agent")
        # 三路估价器 + 文本预处理器
        self.specialist = SpecialistAgent()
        self.frontier = FrontierAgent(collection)
        self.neural_network = NeuralNetworkAgent()
        self.preprocessor = Preprocessor()
        self.log("Ensemble Agent is ready")

    def price(self, description: str) -> float:
        """运行这个集成模型
        询问每个型号的产品定价
        然后使用线性回归模型返回加权价格
        :param description: 产品的描述
        :return: 预估价格"""
        self.log("Running Ensemble Agent - preprocessing text")
        # 先改写成统一短描述，再分别询价
        rewrite = self.preprocessor.preprocess(description)
        self.log(f"Pre-processed text using {self.preprocessor.model_name}")
        specialist = self.specialist.price(rewrite)
        frontier = self.frontier.price(rewrite)
        neural_network = self.neural_network.price(rewrite)
        # 固定权重：前沿模型 0.8，专家与 NN 各 0.1
        combined = frontier * 0.8 + specialist * 0.1 + neural_network * 0.1
        self.log(f"Ensemble Agent complete - returning ${combined:.2f}")
        return combined



In [ ]:
# ========== MessagingAgent：Pushover 推送 + LLM 润色告警文案 ==========

import requests

# Pushover REST 端点（URL 保持原样）
pushover_url = "https://api.pushover.net/1/messages.json"


class MessagingAgent(Agent):
    name = "Messaging Agent"
    color = Agent.WHITE
    # craft_message 用的模型 id
    MODEL = "gpt-5.4"

    def __init__(self):
        """设置此对象以通过 Pushover 执行推送通知，
        或通过 Twilio 发送短信，
        以常数中指定的为准"""
        self.log("Messaging Agent is initializing")
        # 缺省占位串：提醒去配 .env
        self.pushover_user = os.getenv("PUSHOVER_USER", "your-pushover-user-if-not-using-env")
        self.pushover_token = os.getenv("PUSHOVER_TOKEN", "your-pushover-user-if-not-using-env")
        self.log("Messaging Agent has initialized Pushover and Claude")

    def push(self, text):
        """使用 Pushover API 发送推送通知"""
        self.log("Messaging Agent is sending a push notification")
        payload = {
            "user": self.pushover_user,
            "token": self.pushover_token,
            "message": text,
            "sound": "cashregister",
        }
        requests.post(pushover_url, data=payload)

    def alert(self, opportunity: Opportunity):
        """发出有关指定机会的警报"""
        # 直接拼短消息（UI 点选行时走这条）
        text = f"Deal Alert! Price=${opportunity.deal.price:.2f}, "
        text += f"Estimate=${opportunity.estimate:.2f}, "
        text += f"Discount=${opportunity.discount:.2f} :"
        text += opportunity.deal.product_description[:10] + "... "
        text += opportunity.deal.url
        self.push(text)
        self.log("Messaging Agent has completed")

    def craft_message(
        self, description: str, deal_price: float, estimated_true_value: float
    ) -> str:
        # 让模型写 2–3 句兴奋推送（英文 prompt 不翻译）
        user_prompt = "Please summarize this great deal in 2-3 sentences to be sent as an exciting push notification alerting the user about this deal.\n"
        user_prompt += f"Item Description: {description}\nOffered Price: {deal_price}\nEstimated true value: {estimated_true_value}"
        user_prompt += "\n\nRespond only with the 2-3 sentence message which will be used to alert & excite the user about this deal"
        response = completion(
            model=self.MODEL,
            messages=[
                {"role": "user", "content": user_prompt},
            ],
        )
        return response.choices[0].message.content

    def notify(self, description: str, deal_price: float, estimated_true_value: float, url: str):
        """对指定的详细信息发出警报"""
        self.log("Messaging Agent is using Claude to craft the message")
        text = self.craft_message(description, deal_price, estimated_true_value)
        # 正文截断 + URL，避免推送过长
        self.push(text[:200] + "... " + url)
        self.log("Messaging Agent has completed")



In [ ]:
# ========== AutonomousPlanningAgent：Tool loop 编排扫网 / 估价 / 通知 ==========

from typing import Optional, List, Dict

class AutonomousPlanningAgent(Agent):
    name = "Autonomous Planning Agent"
    color = Agent.GREEN
    MODEL = "gpt-5.4"

    def __init__(self, collection):
        """创建该规划器协调的 3 个代理的实例"""
        self.log("Autonomous Planning Agent is initializing")
        self.scanner = ScannerAgent()
        self.ensemble = EnsembleAgent(collection)
        self.messenger = MessagingAgent()
        # 复用笔记本顶部的全局 openai 客户端
        self.openai = openai
        self.memory = None
        self.opportunity = None
        self.log("Autonomous Planning Agent is ready")

    def scan_the_internet_for_bargains(self) -> str:
        """运行工具进行扫描"""
        self.log("Autonomous Planning agent is calling scanner")
        results = self.scanner.scan(memory=self.memory)
        # 给模型 JSON 字符串；无结果则英文提示
        return results.model_dump_json() if results else "No deals found"

    def estimate_true_value(self, description: str) -> str:
        """运行该工具来估计真实值"""
        self.log("Autonomous Planning agent is estimating value via Ensemble Agent")
        estimate = self.ensemble.price(description)
        return f"The estimated true value of {description} is {estimate}"

    def notify_user_of_deal(
        self, description: str, deal_price: float, estimated_true_value: float, url: str
    ) -> Dict:
        """
        Run the tool to notify the user
        """
        # 只允许通知一次：第二次调用直接忽略
        if self.opportunity:
            self.log("Autonomous Planning agent is trying to notify the user a 2nd time; ignoring")
        else:
            self.log("Autonomous Planning agent is notifying user")
            self.messenger.notify(description, deal_price, estimated_true_value, url)
            deal = Deal(product_description=description, price=deal_price, url=url)
            discount = estimated_true_value - deal_price
            self.opportunity = Opportunity(
                deal=deal, estimate=estimated_true_value, discount=discount
            )
        return "Notification sent ok"

    # —— 下面三段是 OpenAI function schema（description 英文给模型看，勿改）——
    scan_function = {
        "name": "scan_the_internet_for_bargains",
        "description": "Returns top bargains scraped from the internet along with the price each item is being offered for",
        "parameters": {
            "type": "object",
            "properties": {},
            "required": [],
            "additionalProperties": False,
        },
    }

    estimate_function = {
        "name": "estimate_true_value",
        "description": "Given the description of an item, estimate how much it is actually worth",
        "parameters": {
            "type": "object",
            "properties": {
                "description": {
                    "type": "string",
                    "description": "The description of the item to be estimated",
                },
            },
            "required": ["description"],
            "additionalProperties": False,
        },
    }

    notify_function = {
        "name": "notify_user_of_deal",
        "description": "Send the user a push notification about the single most compelling deal; only call this one time",
        "parameters": {
            "type": "object",
            "properties": {
                "description": {
                    "type": "string",
                    "description": "The description of the item itself scraped from the internet",
                },
                "deal_price": {
                    "type": "number",
                    "description": "The price offered by this deal scraped from the internet",
                },
                "estimated_true_value": {
                    "type": "number",
                    "description": "The estimated actual value that this is worth",
                },
                "url": {
                    "type": "string",
                    "description": "The URL of this deal as scraped from the internet",
                },
            },
            "required": ["description", "deal_price", "estimated_true_value", "url"],
            "additionalProperties": False,
        },
    }

    def get_tools(self):
        """返回要使用的工具的json"""
        return [
            {"type": "function", "function": self.scan_function},
            {"type": "function", "function": self.estimate_function},
            {"type": "function", "function": self.notify_function},
        ]

    def handle_tool_call(self, message):
        """实际调用与此消息相关的工具"""
        # 名字 → 本地方法
        mapping = {
            "scan_the_internet_for_bargains": self.scan_the_internet_for_bargains,
            "estimate_true_value": self.estimate_true_value,
            "notify_user_of_deal": self.notify_user_of_deal,
        }
        results = []
        for tool_call in message.tool_calls:
            tool_name = tool_call.function.name
            arguments = json.loads(tool_call.function.arguments)
            tool = mapping.get(tool_name)
            result = tool(**arguments) if tool else ""
            results.append({"role": "tool", "content": result, "tool_call_id": tool_call.id})
        return results

    # 类属性上的对话种子（与前面独立 cell 内容一致）
    system_message = "You find great deals on bargain products using your tools, and notify the user of the best bargain."
    user_message = """
    First, use your tool to scan the internet for bargain deals. Then for each deal, use your tool to estimate its true value.
    Then pick the single most compelling deal where the price is much lower than the estimated true value, and use your tool to notify the user.
    Then just reply OK to indicate success.
    """
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_message},
    ]

    def plan(self, memory: List[str] = []) -> Optional[Opportunity]:
        """运行完整的工作流程，为法学硕士提供向用户展示抓取交易的工具
        :param memory: 过去出现过的 URL 列表
        :return: 如果有机会出现，则为机会，否则为无"""
        self.log("Autonomous Planning Agent is kicking off a run")
        self.memory = memory
        self.opportunity = None
        # 拷贝类上的种子 messages，避免污染类属性
        messages = self.messages[:]
        done = False
        while not done:
            response = self.openai.chat.completions.create(
                model=self.MODEL, messages=messages, tools=self.get_tools()
            )
            if response.choices[0].finish_reason == "tool_calls":
                message = response.choices[0].message
                results = self.handle_tool_call(message)
                messages.append(message)
                messages.extend(results)
            else:
                done = True
        reply = response.choices[0].message.content
        self.log(f"Autonomous Planning Agent completed with: {reply}")
        return self.opportunity



In [ ]:
# ========== 试跑：构造规划代理并执行一轮 plan ==========

agent = AutonomousPlanningAgent(collection)
agent.plan()


In [ ]:
# ========== DealAgentFramework：memory 持久化 + 可选 t-SNE 可视化数据 ==========

import sys
from typing import List
from sklearn.manifold import TSNE
import numpy as np

# 用于记录的颜色（Framework 日志用蓝底白字）
BG_BLUE = "\033[44m"
WHITE = "\033[37m"
RESET = "\033[0m"

# 绘图的颜色：类别名 → 固定调色板索引
CATEGORIES = [
    "Appliances",
    "Automotive",
    "Cell_Phones_and_Accessories",
    "Electronics",
    "Musical_Instruments",
    "Office_Products",
    "Tools_and_Home_Improvement",
    "Toys_and_Games",
]
COLORS = ["red", "blue", "brown", "orange", "yellow", "green", "purple", "cyan"]


def init_logging():
    # 把 INFO 日志打到 stdout，带时间戳
    root = logging.getLogger()
    root.setLevel(logging.INFO)

    handler = logging.StreamHandler(sys.stdout)
    handler.setLevel(logging.INFO)
    formatter = logging.Formatter(
        "[%(asctime)s] [Agents] [%(levelname)s] %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S %z",
    )
    handler.setFormatter(formatter)
    root.addHandler(handler)


class DealAgentFramework:
    DB = "products_vectorstore"
    MEMORY_FILENAME = "memory.json"

    def __init__(self):
        init_logging()
        client = chromadb.PersistentClient(path=self.DB)
        self.memory = self.read_memory()
        self.collection = client.get_or_create_collection("products")
        self.planner = None

    def init_agents_as_needed(self):
        # 懒加载：只在第一次 run / UI 需要时建 AutonomousPlanningAgent
        if not self.planner:
            self.log("Initializing Agent Framework")
            self.planner = AutonomousPlanningAgent(self.collection)
            self.log("Agent Framework is ready")

    def read_memory(self) -> List[Opportunity]:
        if os.path.exists(self.MEMORY_FILENAME):
            with open(self.MEMORY_FILENAME, "r") as file:
                data = json.load(file)
            opportunities = [Opportunity(**item) for item in data]
            return opportunities
        return []

    def write_memory(self) -> None:
        data = [opportunity.model_dump() for opportunity in self.memory]
        with open(self.MEMORY_FILENAME, "w") as file:
            json.dump(data, file, indent=2)

    @classmethod
    def reset_memory(cls) -> None:
        # 调试用：只保留前 2 条历史
        data = []
        if os.path.exists(cls.MEMORY_FILENAME):
            with open(cls.MEMORY_FILENAME, "r") as file:
                data = json.load(file)
        truncated = data[:2]
        with open(cls.MEMORY_FILENAME, "w") as file:
            json.dump(truncated, file, indent=2)

    def log(self, message: str):
        text = BG_BLUE + WHITE + "[Agent Framework] " + message + RESET
        logging.info(text)

    def run(self) -> List[Opportunity]:
        self.init_agents_as_needed()
        logging.info("Kicking off Planning Agent")
        result = self.planner.plan(memory=self.memory)
        logging.info(f"Planning Agent has completed and returned: {result}")
        if result:
            self.memory.append(result)
            self.write_memory()
        return self.memory

    @classmethod
    def get_plot_data(cls, max_datapoints=2000):
        # 从 Chroma 取 embedding，t-SNE 降到 3D 供可视化
        client = chromadb.PersistentClient(path=cls.DB)
        collection = client.get_or_create_collection("products")
        result = collection.get(
            include=["embeddings", "documents", "metadatas"], limit=max_datapoints
        )
        vectors = np.array(result["embeddings"])
        documents = result["documents"]
        categories = [metadata["category"] for metadata in result["metadatas"]]
        colors = [COLORS[CATEGORIES.index(c)] for c in categories]
        tsne = TSNE(n_components=3, random_state=42, n_jobs=-1)
        reduced_vectors = tsne.fit_transform(vectors)
        return documents, reduced_vectors, colors


if __name__ == "__main__":
    DealAgentFramework().run()



In [ ]:
# ========== 导入 Gradio：下一格搭「价格竞猜」看板 ==========

import gradio as gr


In [ ]:
# ========== Gradio：展示机会表，点击行触发 messenger.alert ==========

# 框架单例：内部会懒加载 AutonomousPlanningAgent
agent_framework = DealAgentFramework()
agent_framework.init_agents_as_needed()

with gr.Blocks(title="The Price is Right", fill_width=True) as ui:

    # 占位示例行，让表格首屏非空
    initial_deal = Deal(product_description="Example description", price=100.0, url="https://cnn.com")
    initial_opportunity = Opportunity(deal=initial_deal, estimate=200.0, discount=100.0)
    opportunities = gr.State([initial_opportunity])

    def get_table(opps):
        # Opportunity → Dataframe 五行
        return [[opp.deal.product_description, opp.deal.price, opp.estimate, opp.discount, opp.deal.url] for opp in opps]

    def do_select(opportunities, selected_index: gr.SelectData):
        # 点中的行 → 对该 Opportunity 发 Pushover/告警
        row = selected_index.index[0]
        opportunity = opportunities[row]
        agent_framework.planner.messenger.alert(opportunity)

    with gr.Row():
        gr.Markdown('<div style="text-align: center;font-size:24px">"The Price is Right" - Deal Hunting Agentic AI</div>')
    with gr.Row():
        gr.Markdown('<div style="text-align: center;font-size:14px">Deals surfaced so far:</div>')
    with gr.Row():
        opportunities_dataframe = gr.Dataframe(
            headers=["Description", "Price", "Estimate", "Discount", "URL"],
            wrap=True,
            column_widths=[4, 1, 1, 1, 2],
            row_count=10,
            col_count=5,
            max_height=400,
        )

    # 页面加载时用 State 填表
    ui.load(get_table, inputs=[opportunities], outputs=[opportunities_dataframe])
    opportunities_dataframe.select(do_select, inputs=[opportunities], outputs=[])

ui.launch(inbrowser=True)
